# Embeddings

Контекст помогает понять значение слова не хуже словаря. Именно на этом основана **дистрибутивная гипотеза**: слова, которые встречаются в похожих контекстах, обычно имеют похожие значения.

В этой лекции мы пройдём путь от простых **count-based** представлений до **dense embeddings** вроде `word2vec` и `GloVe`, а затем обсудим их свойства, ограничения и способы оценки.


### План лекции

1. Лексическая семантика и дистрибутивная гипотеза.
2. Почему `one-hot` недостаточно.
3. Count-based embeddings: матрица совместных встречаемостей.
4. Косинусное сходство.
5. PMI / PPMI как улучшение сырых счётчиков.
6. Dense embeddings: `word2vec`, `skip-gram`, negative sampling.
7. `GloVe`, аналогии, bias, оценка моделей.

### Что важно унести с собой

- Эмбеддинг — это векторное представление слова.
- Вектор строится не вручную, а обучается по статистике контекстов.
- Близость векторов часто отражает семантическую близость слов.
- У эмбеддингов есть не только сильные стороны, но и ограничения: полисемия, bias, зависимость от корпуса.


### 1. Лексическая семантика и дистрибутивная гипотеза

В главе сначала обсуждаются базовые понятия **lexical semantics**:

- **lemma** — словарная форма слова (`mouse`, `sing`);
- **wordform** — конкретная словоформа (`mice`, `sang`);
- **sense** — отдельное значение многозначного слова;
- **synonymy** — близость значений (`car`, `automobile`);
- **similarity** — семантическое сходство (`cat`, `dog`);
- **relatedness** — ассоциативная связь (`coffee`, `cup`);
- **connotation** — эмоциональная окраска (`happy` vs `dreary`).

Ключевая мысль главы формулируется фразой J. R. Firth:

> *Words which occur in similar contexts tend to have similar meanings.*

Это и есть **дистрибутивная гипотеза**. Она не говорит, что смысл слова полностью равен контексту, но говорит, что по контексту можно много узнать о значении.

Пример из главы: если мы не знаем слово `ongchoi`, но видим его рядом с `garlic`, `rice`, `leaves`, `salty`, то можем предположить, что это что-то вроде листового овоща.


### 2. Почему `one-hot` encoding недостаточно

`One-hot` encoding полезен как технический способ кодирования категорий, но почти бесполезен как модель смысла слова.

Если словарь состоит из `N` слов, то каждое слово кодируется вектором длины `N`, где только одна позиция равна `1`.

Проблема в том, что такие векторы:

- не содержат информации о сходстве слов;
- одинаково далеки друг от друга;
- не умеют отражать контекст;
- не обобщают на похожие слова.

Например, в `one-hot` представлении слова `cat` и `dog` не ближе друг к другу, чем `cat` и `democracy`.

Именно поэтому в NLP нужны **representation learning**-подходы: модель должна **сама выучить** полезное представление слова по данным.


In [ ]:
import numpy as np
import pandas as pd

vocab = ["cat", "dog", "democracy", "computer"]
one_hot = np.eye(len(vocab), dtype=int)

df_one_hot = pd.DataFrame(one_hot, index=vocab, columns=vocab)
print("One-hot представление слов:")
display(df_one_hot)

print("\nСкалярные произведения разных one-hot векторов:")
print("cat · dog =", int(one_hot[0] @ one_hot[1]))
print("cat · democracy =", int(one_hot[0] @ one_hot[2]))
print("dog · computer =", int(one_hot[1] @ one_hot[3]))


### Вывод

Во всех трёх случаях скалярное произведение равно `0`. Для `one-hot` кодирования любые два разных слова одинаково несхожи.

Это хороший контраст с эмбеддингами: в хорошем векторном пространстве `cat` и `dog` должны быть ближе, чем `cat` и `computer`.


### 3. Count-based embeddings: слово как вектор контекстов

Самая простая идея из главы: представить слово через то, **какие слова встречаются рядом с ним**.

Строится **word-context matrix**:

- строки — целевые слова;
- столбцы — контекстные слова;
- ячейка — сколько раз контекстное слово встретилось рядом с целевым.

Обычно используется контекстное окно, например `±2`, `±4` или `±5` слов.

Это даёт **sparse embeddings**:

- векторы длинные;
- большинство координат равны нулю;
- каждая координата имеет интерпретацию: это конкретное контекстное слово.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Self-contained illustration of a small co-occurrence vector.
vector = [1, 0, 1]
labels = ["a", "computer", "pie"]
x_positions = [0, 1, 2]

fig, ax = plt.subplots(figsize=(8, 2.8))

for x, value, label in zip(x_positions, vector, labels):
    rect = Rectangle((x, 0), 1, 1, facecolor="#dbeafe", edgecolor="#1f2937", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.5, 0.62, label, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(x + 0.5, 0.28, str(value), ha="center", va="center", fontsize=14, color="#b91c1c")

highlight = Rectangle((0, 0), 3, 1, fill=False, edgecolor="#ef4444", linewidth=2.5)
ax.add_patch(highlight)
ax.text(-0.15, 0.5, "cherry", ha="right", va="center", fontsize=13, fontweight="bold")
ax.text(1.5, -0.22, "Example vector: cherry -> [1, 0, 1]", ha="center", va="top", fontsize=11)

ax.set_xlim(-0.7, 3.2)
ax.set_ylim(-0.4, 1.2)
ax.axis("off")
plt.show()


In [ ]:
from collections import Counter

corpus = [
    "cherry pie is a traditional dessert",
    "strawberry pie tastes sweet and bright",
    "digital computer stores data quickly",
    "information and data live inside a computer",
]

window_size = 2
target_words = ["cherry", "strawberry", "digital", "information"]
context_words = ["a", "computer", "pie", "data", "traditional", "sweet"]

cooc = {target: Counter({context: 0 for context in context_words}) for target in target_words}

for sentence in corpus:
    tokens = sentence.split()
    for i, token in enumerate(tokens):
        if token not in target_words:
            continue
        left = max(0, i - window_size)
        right = min(len(tokens), i + window_size + 1)
        context = tokens[left:i] + tokens[i + 1:right]
        for ctx in context:
            if ctx in context_words:
                cooc[token][ctx] += 1

cooc_df = pd.DataFrame.from_dict(cooc, orient="index")
cooc_df = cooc_df[context_words]
print("Co-occurrence matrix (слово-контекст):")
display(cooc_df)


### Что показывает такая матрица

Если два слова получают похожие строки в матрице совместных встречаемостей, мы считаем их семантически близкими.

Например:

- `cherry` и `strawberry` будут близки, если часто стоят рядом с `pie`, `sweet`, `dessert`;
- `digital` и `information` будут близки, если рядом часто встречаются `computer`, `data`, `system`.

У такой модели есть плюс: она очень прозрачна. Мы буквально видим, **из каких контекстов состоит значение**.

Но есть и минусы:

- векторы очень разреженные;
- размерность примерно равна размеру словаря;
- сырые счётчики чувствительны к частотности слов.


### 4. Косинусное сходство

Чтобы сравнить два слова, недостаточно просто смотреть на сырые счётчики. Частые слова имеют более длинные векторы и из-за этого могут казаться искусственно похожими.

Поэтому стандартная мера в NLP — **cosine similarity**:

$$
\text{cosine}(v, w) = \frac{v \cdot w}{\|v\|\|w\|}
$$

Интуиция простая:

- если векторы направлены почти в одну сторону, косинус близок к `1`;
- если ортогональны, косинус близок к `0`;
- если направлены противоположно, косинус меньше `0`.

Для count-based моделей косинус почти всегда полезнее, чем обычный dot product.


In [ ]:
import matplotlib.pyplot as plt

digital = np.array([1683, 1670])
information = np.array([3982, 3325])

fig, ax = plt.subplots(figsize=(6, 5))
ax.arrow(0, 0, digital[0], digital[1], length_includes_head=True,
         head_width=120, head_length=180, linewidth=2.5, color="#16a34a", alpha=0.9)
ax.arrow(0, 0, information[0], information[1], length_includes_head=True,
         head_width=120, head_length=180, linewidth=2.5, color="#2563eb", alpha=0.9)

ax.scatter([digital[0], information[0]], [digital[1], information[1]],
           color=["#16a34a", "#2563eb"], s=45, zorder=3)
ax.text(digital[0] * 0.62, digital[1] * 0.92, "digital\n[1683, 1670]", color="#16a34a", fontsize=11)
ax.text(information[0] * 0.78, information[1] * 0.95, "information\n[3982, 3325]", color="#1d4ed8", fontsize=11)

ax.set_xlabel("data", fontsize=12, fontweight="bold")
ax.set_ylabel("computer", fontsize=12, fontweight="bold")
ax.set_title("Geometric intuition: two word vectors in 2D", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.35)
ax.set_xlim(0, 4500)
ax.set_ylim(0, 3800)
ax.set_aspect("equal", adjustable="box")
plt.show()


In [ ]:
def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

pairs = [
    ("cherry", "strawberry"),
    ("digital", "information"),
    ("cherry", "digital"),
]

for left, right in pairs:
    score = cosine_similarity(cooc_df.loc[left].values, cooc_df.loc[right].values)
    print(f"cos({left}, {right}) = {score:.3f}")


### Интерпретация результатов

Обычно мы ожидаем, что:

- `cherry` будет ближе к `strawberry`, чем к `digital`;
- `digital` будет ближе к `information`, чем к `cherry`.

Если это так, значит распределительная статистика уже начала отражать семантику.


### 5. Почему сырые счётчики не идеальны: PMI и PPMI

Сырые co-occurrence counts — это только первый шаг. Проблема в том, что частые слова вроде `the`, `of`, `and` встречаются почти со всеми и засоряют статистику.

Один из классических способов улучшить матрицу — **PMI** (*pointwise mutual information*):

$$
PMI(w, c) = \log_2 \frac{P(w, c)}{P(w)P(c)}
$$

Идея такая:

- если слово и контекст встречаются вместе **чаще, чем ожидалось случайно**, PMI положителен;
- если примерно как ожидалось, PMI около нуля;
- если реже ожидания, PMI отрицателен.

На практике часто используют **PPMI**:

$$
PPMI(w, c) = \max(PMI(w, c), 0)
$$

То есть просто отбрасывают отрицательные значения.

В литературе это один из самых сильных классических baseline-подходов для sparse embeddings.


In [ ]:
count_matrix = cooc_df.values.astype(float)
total = count_matrix.sum()
row_sums = count_matrix.sum(axis=1, keepdims=True)
col_sums = count_matrix.sum(axis=0, keepdims=True)

with np.errstate(divide="ignore", invalid="ignore"):
    p_wc = count_matrix / total
    p_w = row_sums / total
    p_c = col_sums / total
    pmi = np.log2(p_wc / (p_w @ p_c))
    pmi[np.isinf(pmi)] = np.nan
    ppmi = np.maximum(np.nan_to_num(pmi, nan=0.0, neginf=0.0, posinf=0.0), 0.0)

ppmi_df = pd.DataFrame(ppmi, index=cooc_df.index, columns=cooc_df.columns)
print("PPMI matrix:")
display(ppmi_df.round(2))


### Что изменилось после PPMI

PPMI усиливает **информативные связи** и ослабляет влияние сверхчастотных слов.

Именно поэтому в исследованиях по vector semantics часто сравнивают:

- raw counts;
- tf-idf или похожие взвешивания;
- PMI / PPMI;
- dense embeddings.

Есть важный результат из более поздней литературы: Skip-gram with Negative Sampling можно интерпретировать как неявную факторизацию специальным образом сдвинутой PPMI-матрицы. Это хорошая интуитивная связь между count-based и prediction-based подходами.


### 6. Sparse vs Dense embeddings

Теперь можно сформулировать главное различие.

**Sparse vectors**:

- размерность велика (`|V|`);
- координаты интерпретируемы;
- строятся по счётчикам или их взвешенным версиям.

**Dense vectors**:

- размерность обычно `50..1000`;
- координаты напрямую не интерпретируются;
- обучаются так, чтобы помогать предсказывать контекст.

Dense embeddings обычно лучше работают в downstream-задачах, потому что:

- компактнее;
- лучше обобщают;
- позволяют уловить скрытые закономерности и похожесть между словами, даже если контексты совпадают не буквально.


### 7. `word2vec`: идея `skip-gram`

В главе рассматривается `word2vec`, а точнее наиболее известный вариант: **skip-gram with negative sampling (SGNS)**.

Основная идея:

- берём целевое слово `w`;
- смотрим на реальные слова контекста `c`, которые стоят рядом;
- учим модель отличать реальные пары `(w, c)` от случайных пар `(w, noise)`.

Модель не хранит “значение слова” явно. Она решает побочную задачу: **предсказать, какое слово могло бы встретиться рядом**. Но в процессе обучения веса модели превращаются в хорошие эмбеддинги.

Это классический пример **self-supervised learning**.

С вероятностной точки зрения модель использует сигмоиду:

$$
P(+|w, c) = \sigma(w \cdot c)
$$

где большой dot product означает: “эта пара похожа на настоящий контекст”.


In [ ]:
def generate_skipgram_pairs(tokens, window_size=2):
    pairs = []
    for i, target in enumerate(tokens):
        left = max(0, i - window_size)
        right = min(len(tokens), i + window_size + 1)
        for j in range(left, right):
            if i != j:
                pairs.append((target, tokens[j]))
    return pairs

sentence = "apricot jam on warm bread tastes sweet".split()
positive_pairs = generate_skipgram_pairs(sentence, window_size=2)

print("Положительные пары (target, context):")
for pair in positive_pairs[:12]:
    print(pair)


### Negative sampling

Если бы мы пытались для каждого слова считать вероятности по всему словарю, обучение было бы дорогим.

`Negative sampling` упрощает задачу:

- положительные примеры берём из настоящего текста;
- отрицательные получаем случайной подстановкой слов из словаря;
- модель учится повышать score для настоящих контекстов и понижать для шумовых.

В оригинальной работе Mikolov et al. (2013) это дало быстрое и качественное обучение. Глава также подчёркивает, что для шума используется не просто униграммное распределение, а обычно вероятность вида `P(w)^0.75`.


In [ ]:
rng = np.random.default_rng(42)

vocab = sorted(set(sentence + ["computer", "city", "banana", "network"]))
word_to_id = {word: i for i, word in enumerate(vocab)}
dim = 5
W = rng.normal(0, 0.2, size=(len(vocab), dim))
C = rng.normal(0, 0.2, size=(len(vocab), dim))

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score(target, context):
    w = W[word_to_id[target]]
    c = C[word_to_id[context]]
    return float(sigmoid(np.dot(w, c)))

sample_positive = ("apricot", "jam")
sample_negative = ("apricot", "network")

print(f"P(+ | {sample_positive}) = {score(*sample_positive):.4f}")
print(f"P(+ | {sample_negative}) = {score(*sample_negative):.4f}")


### Как это интерпретировать

До обучения вероятности почти случайны. После обучения модель должна сделать так, чтобы:

- реальные пары вроде `("apricot", "jam")` получали высокий score;
- шумовые пары вроде `("apricot", "network")` — низкий.

Именно итоговые веса матриц `W` и `C` становятся эмбеддингами.

Обычно после обучения в качестве итогового вектора слова берут либо строку из `W`, либо комбинацию `W` и `C`.


### 8. `GloVe`: глобальная статистика корпуса

`word2vec` относится к **prediction-based** подходам: модель учится по задаче предсказания контекста.

`GloVe` (*Global Vectors for Word Representation*, Pennington, Socher, Manning, 2014) делает акцент на **global co-occurrence statistics**.

Идея `GloVe`:

- сначала собирается глобальная матрица совместных встречаемостей;
- затем обучаются dense-векторы, которые хорошо объясняют логарифмы этих совместных частот;
- таким образом объединяются сильные стороны count-based и prediction-based подходов.

Практически полезная интуиция:

- `word2vec` учится по локальным окнам и задаче классификации;
- `GloVe` опирается на агрегированную статистику по всему корпусу.

Обе модели создают **static embeddings**: у слова один вектор на все случаи употребления.


### 9. Семантические свойства эмбеддингов

У эмбеддингов есть несколько замечательных свойств.

#### 9.1. Близость отражает семантическое сходство

Если модель обучена хорошо, то ближайшие соседи слова часто оказываются либо:

- семантически похожими словами;
- либо тематически связанными словами.

При этом размер контекстного окна влияет на результат:

- маленькое окно чаще даёт **syntactic / substitutable similarity**;
- большое окно чаще даёт **topical relatedness**.

#### 9.2. Аналогии и линейные направления

Классический пример:

$$
king - man + woman \approx queen
$$

Это не магия, а признак того, что некоторые отношения в векторном пространстве кодируются как направления.


In [ ]:
embeddings = {
    "man": np.array([0.0, 1.0, 0.0]),
    "woman": np.array([0.0, -1.0, 0.0]),
    "king": np.array([1.0, 1.0, 0.0]),
    "queen": np.array([1.0, -1.0, 0.0]),
}

query = embeddings["king"] - embeddings["man"] + embeddings["woman"]

for word, vec in embeddings.items():
    print(word, cosine_similarity(query, vec))


### Осторожно с аналогиями

Аналогии красивы как демонстрация, но их нельзя переоценивать.

Глава специально предупреждает:

- метод `b - a + a*` работает не для всех отношений;
- он чувствителен к частотности слов и качеству корпуса;
- иногда возвращает одно из исходных слов или его форму вместо правильного ответа.

То есть аналогии — интересное свойство пространства, но не универсальный тест “понимания смысла”.


### 10. Bias и ограничения эмбеддингов

Очень важная часть главы: эмбеддинги наследуют свойства обучающего корпуса, включая **социальные стереотипы и bias**.

Если в текстах систематически встречаются перекосы, то модель может их воспроизводить. Это опасно, потому что вектора затем используются в downstream-системах:

- ранжировании;
- поиске;
- классификации;
- рекомендательных и диалоговых системах.

Кроме bias, есть и другие ограничения:

- **полисемия**: у слова `bank` один вектор, хотя значений несколько;
- **статичность**: одно представление для всех контекстов;
- **зависимость от корпуса**: другой корпус — другое пространство;
- **нестабильность**: обучение может давать немного разные результаты при разных инициализациях.

Именно поэтому позже в NLP появились **contextual embeddings** (`ELMo`, `BERT` и др.), где вектор слова зависит от конкретного контекста.


### 11. Как оценивают эмбеддинги

Глава выделяет два основных типа оценки.

#### 11.1. Intrinsic evaluation

Смотрим на внутренние свойства пространства:

- similarity datasets: `WordSim-353`, `SimLex-999`;
- analogy tasks;
- contextual similarity datasets вроде `SCWS` или `WiC`.

#### 11.2. Extrinsic evaluation

Смотрим, улучшают ли эмбеддинги реальную NLP-задачу:

- классификацию текста;
- NER;
- question answering;
- summarization;
- retrieval.

С практической точки зрения **extrinsic evaluation важнее**, потому что красивое пространство ещё не гарантирует пользу в приложении.


### 12. Историческая перспектива

Глава хорошо показывает, что embeddings не возникли внезапно.

Короткая линия развития выглядит так:

- 1950-е: Firth, Harris, Joos формулируют распределительный взгляд на значение;
- 1957: Osgood предлагает представлять значения как точки в семантическом пространстве;
- 1990-е: LSA / SVD и другие матричные методы;
- 2003: Bengio et al. показывают силу распределённых представлений в нейросетевой языковой модели;
- 2013: Mikolov et al. делают `word2vec` массово популярным;
- 2014: Pennington et al. предлагают `GloVe`.

Идея одна и та же: **значение можно изучать через структуру распределений в тексте**.


### 13. Главное резюме главы

- Слово можно представить как точку в векторном пространстве.
- Простые векторы строятся по матрице совместных встречаемостей.
- Косинусное сходство позволяет сравнивать слова по направлению их векторов.
- Взвешивания вроде PMI / PPMI улучшают sparse-представления.
- `word2vec` учит dense-векторы через задачу предсказания контекста.
- `GloVe` строит dense-векторы на глобальной статистике корпуса.
- Эмбеддинги полезны, но не идеальны: у них есть bias, проблемы с полисемией и зависимостью от корпуса.


### 14. Упражнения

1. Объясните своими словами разницу между **similarity** и **relatedness**. Приведите по одному примеру.
2. Почему для сравнения слов обычно лучше использовать **cosine similarity**, а не сырой dot product?
3. Что именно делает `negative sampling` и зачем он нужен?
4. Почему `word2vec` и `GloVe` называют **static embeddings**?
5. В чём главная проблема одного фиксированного вектора для слова `mouse` или `bank`?


### 15. Мини-разбор ответов

1. **Similarity** — слова похожи по смыслу (`car`, `automobile`; `cat`, `dog`). **Relatedness** — слова связаны ассоциативно, но не обязаны быть похожи (`coffee`, `cup`).
2. Dot product зависит от длины вектора, а косинус нормирует по длине и лучше отражает именно направление.
3. `Negative sampling` подаёт модели случайные “ложные” контексты, чтобы обучение отличало реальные соседства слов от шума.
4. Потому что каждому слову сопоставляется один и тот же вектор вне зависимости от конкретного предложения.
5. Полисемия: у одного слова может быть несколько значений, а один статический вектор смешивает их в одну точку.


### Источники и что было использовано сверх главы

Помимо самой главы `Chapter_05_Embeddings.pdf`, в объяснения были встроены идеи из первоисточников:

- Bengio et al. (2003), *A Neural Probabilistic Language Model*.
- Mikolov et al. (2013), *Distributed Representations of Words and Phrases and their Compositionality*.
- Pennington et al. (2014), *GloVe: Global Vectors for Word Representation*.

Это помогло точнее связать count-based и prediction-based подходы и аккуратно пояснить роль `negative sampling` и глобальной статистики в `GloVe`.
